# NB07: Cohort Classification & Synthesis

Final integration notebook. Combines outputs from NB01–NB06 into composite
beneficial/pathogenic/dual-nature/neutral classifications, validates against
known organisms, generates genus-level dossiers, and produces a synthesis
figure summarizing all five project hypotheses.

**Inputs** (from prior notebooks):
- `species_compartment.csv` (NB01)
- `species_marker_matrix.csv`, `species_cohort_markers.csv` (NB02)
- `enrichment_results.csv`, `novel_plant_markers.csv` (NB03)
- `compartment_profiles.csv`, `genus_profiles.csv` (NB04)
- `genomic_architecture.csv` (NB05)
- `complementarity_network.csv` (NB06)

**Outputs**:
- `data/cohort_assignments.csv`
- `data/genus_dossiers.csv`
- `figures/synthesis_overview.png`

In [ ]:
import os, re, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore', category=FutureWarning)

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))
elif os.path.exists(os.path.join(_here, 'projects', 'plant_microbiome_ecotypes')):
    REPO = _here
else:
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))
PROJECT = os.path.join(REPO, 'projects', 'plant_microbiome_ecotypes')
DATA = os.path.join(PROJECT, 'data')
FIGURES = os.path.join(PROJECT, 'figures')
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')

## 1. Load all prior notebook outputs

In [ ]:
# --- NB01: species compartment assignments ---
species_comp = pd.read_csv(os.path.join(DATA, 'species_compartment.csv'))
print(f'species_compartment: {len(species_comp):,} species')

# --- NB02: marker gene survey ---
species_markers = pd.read_csv(os.path.join(DATA, 'species_marker_matrix.csv'))
cohort_markers = pd.read_csv(os.path.join(DATA, 'species_cohort_markers.csv'))
print(f'species_marker_matrix: {len(species_markers):,} species')
print(f'species_cohort_markers: {len(cohort_markers):,} species')

# --- NB03: enrichment analysis ---
enrichment = pd.read_csv(os.path.join(DATA, 'enrichment_results.csv'))
novel_markers = pd.read_csv(os.path.join(DATA, 'novel_plant_markers.csv'))
print(f'enrichment_results: {len(enrichment):,} rows')
print(f'novel_plant_markers: {len(novel_markers):,} markers')

# --- NB04: compartment & genus profiles ---
comp_profiles = pd.read_csv(os.path.join(DATA, 'compartment_profiles.csv'))
genus_profiles = pd.read_csv(os.path.join(DATA, 'genus_profiles.csv'))
print(f'compartment_profiles: {len(comp_profiles):,} rows')
print(f'genus_profiles: {len(genus_profiles):,} genera')

# --- NB05: genomic architecture ---
genomic_arch = pd.read_csv(os.path.join(DATA, 'genomic_architecture.csv'))
print(f'genomic_architecture: {len(genomic_arch):,} rows')

# --- NB06: complementarity network ---
comp_network = pd.read_csv(os.path.join(DATA, 'complementarity_network.csv'))
print(f'complementarity_network: {len(comp_network):,} edges')

## 2. Composite scoring for cohort classification

Build five sub-scores and combine them into a composite classification.

| Sub-score | Source | Rationale |
|-----------|--------|----------|
| PGP score | NB02 markers | Weighted count of beneficial gene categories |
| Pathogenicity score | NB02 markers | Weighted count of virulence gene categories |
| Metabolic capability | NB04 GapMind | Breadth of complete metabolic pathways |
| Genomic architecture | NB05 | Core fraction of beneficial genes (stability) |
| Co-occurrence | NB06 | Average complementarity with known PGPB partners |

In [ ]:
# ---------------------------------------------------------------
# 2a. PGP score: weighted sum of PGP marker categories
# ---------------------------------------------------------------
#  nif = 3 (nitrogen fixation, high ecological impact)
#  siderophore = 2
#  acc_deaminase = 2
#  all other PGP categories = 1

PGP_WEIGHTS = {
    'nitrogen_fixation': 3,
    'siderophore': 2,
    'acc_deaminase': 2,
    'phosphate_solubilization': 1,
    'iaa_biosynthesis': 1,
    'hydrogen_cyanide': 1,
    'acetoin_butanediol': 1,
    'dapg_biocontrol': 1,
    'phenazine': 1,
    'biofilm': 1,
}

# Identify PGP-related columns in the marker matrix
# The marker matrix has raw counts per functional_category as columns
pgp_category_cols = [c for c in species_markers.columns
                     if c in PGP_WEIGHTS]

def compute_pgp_score(row):
    """Weighted sum of PGP marker presence."""
    score = 0
    for cat in pgp_category_cols:
        if row.get(cat, 0) > 0:
            score += PGP_WEIGHTS.get(cat, 1)
    return score

species_markers['pgp_score_raw'] = species_markers.apply(compute_pgp_score, axis=1)

print(f'PGP category columns found: {len(pgp_category_cols)}')
print(f'  columns: {pgp_category_cols}')
print(f'PGP score: mean={species_markers["pgp_score_raw"].mean():.2f}, '
      f'max={species_markers["pgp_score_raw"].max()}')

In [ ]:
# ---------------------------------------------------------------
# 2b. Pathogenicity score: weighted sum of pathogen marker categories
# ---------------------------------------------------------------
#  T3SS = 3 (hallmark plant pathogen secretion)
#  effector = 3
#  CWDE = 2 (cell wall degrading enzymes)
#  T6SS = 1 (context-dependent, also in biocontrol)
#  all other pathogen categories = 1

PATHOGEN_WEIGHTS = {
    't3ss': 3,
    't3ss_needle': 3,
    't3ss_prgH': 3,
    't3ss_secretin': 2,
    't3ss_product': 3,
    't3ss_yscC': 3,
    't3ss_yscJ': 3,
    't3ss_yscN': 3,
    't3ss_yscV': 3,
    't3ss_yscQ': 3,
    't3ss_yscR': 3,
    'effector': 3,
    'cwde_pectinase': 2,
    'cwde_cellulase': 2,
    'pectate_lyase': 2,
    'pectate_lyase_3': 2,
    'cellulase': 2,
    'coronatine_toxin': 2,
    'phytotoxin': 2,
    't4ss': 1,
    'virb8_t4ss': 1,
    't6ss': 1,
    't6ss_hcp': 1,
    't6ss_vgrg': 1,
    't6ss_product': 1,
    't2ss': 1,
    'other_pathogenic': 1,
}

pathogen_category_cols = [c for c in species_markers.columns
                          if c in PATHOGEN_WEIGHTS]

def compute_pathogen_score(row):
    """Weighted sum of pathogen marker presence."""
    score = 0
    for cat in pathogen_category_cols:
        if row.get(cat, 0) > 0:
            score += PATHOGEN_WEIGHTS.get(cat, 1)
    return score

species_markers['pathogen_score_raw'] = species_markers.apply(compute_pathogen_score, axis=1)

print(f'Pathogen category columns found: {len(pathogen_category_cols)}')
print(f'  columns: {pathogen_category_cols}')
print(f'Pathogen score: mean={species_markers["pathogen_score_raw"].mean():.2f}, '
      f'max={species_markers["pathogen_score_raw"].max()}')

In [ ]:
# ---------------------------------------------------------------
# 2c. Metabolic capability score from genus_profiles (NB04)
# ---------------------------------------------------------------
# Use GapMind pathway completeness breadth if available in genus_profiles,
# otherwise fall back to the number of distinct functional categories.

# Try to extract metabolic breadth from genus_profiles
print('genus_profiles columns:', list(genus_profiles.columns))

# Determine which column best represents metabolic breadth
metabolic_cols = [c for c in genus_profiles.columns
                  if 'gapmind' in c.lower() or 'pathway' in c.lower()
                  or 'metabol' in c.lower() or 'breadth' in c.lower()
                  or 'completeness' in c.lower()]
print(f'Metabolic breadth candidate columns: {metabolic_cols}')

if metabolic_cols:
    metabolic_col = metabolic_cols[0]
    print(f'Using column: {metabolic_col}')
else:
    # Fall back: count number of non-zero functional annotations per genus
    print('No explicit metabolic breadth column; will derive from marker counts.')
    metabolic_col = None

In [ ]:
# ---------------------------------------------------------------
# 2d. Build genus-level aggregation key
# ---------------------------------------------------------------
# species_markers uses gtdb_species_clade_id; we need genus from cohort_markers

# Merge genus info from cohort_markers
scoring_df = species_markers[['gtdb_species_clade_id',
                               'pgp_score_raw', 'pathogen_score_raw',
                               'n_pgp_functions', 'n_pathogen_functions']].copy()

# Add taxonomy and compartment from cohort_markers
tax_cols_needed = ['gtdb_species_clade_id', 'genus', 'phylum', 'dominant_compartment']
tax_available = [c for c in tax_cols_needed if c in cohort_markers.columns]
scoring_df = scoring_df.merge(
    cohort_markers[tax_available].drop_duplicates('gtdb_species_clade_id'),
    on='gtdb_species_clade_id', how='left'
)

# If genus not in cohort_markers, try species_comp
if 'genus' not in scoring_df.columns or scoring_df['genus'].isna().all():
    scoring_df = scoring_df.drop(columns=['genus'], errors='ignore')
    scoring_df = scoring_df.merge(
        species_comp[['gtdb_species_clade_id', 'genus', 'dominant_compartment']].drop_duplicates('gtdb_species_clade_id'),
        on='gtdb_species_clade_id', how='left'
    )

print(f'Scoring dataframe: {len(scoring_df):,} species')
print(f'  with genus label: {scoring_df["genus"].notna().sum():,}')
print(f'  with compartment: {scoring_df["dominant_compartment"].notna().sum():,}')

In [ ]:
# ---------------------------------------------------------------
# 2e. Genomic architecture score from NB05
# ---------------------------------------------------------------
# Core fraction of beneficial genes: species with higher core fraction
# for PGP genes have more stable beneficial potential.

print('genomic_architecture columns:', list(genomic_arch.columns))

# Identify core fraction columns
core_frac_cols = [c for c in genomic_arch.columns
                  if 'core' in c.lower() and ('frac' in c.lower() or 'pct' in c.lower()
                  or 'ratio' in c.lower() or 'proportion' in c.lower())]
print(f'Core fraction candidate columns: {core_frac_cols}')

# Identify the species/genus key
arch_key = None
for candidate in ['gtdb_species_clade_id', 'species', 'genus']:
    if candidate in genomic_arch.columns:
        arch_key = candidate
        break
print(f'Architecture join key: {arch_key}')

if core_frac_cols and arch_key:
    arch_score_col = core_frac_cols[0]
    arch_merge = genomic_arch[[arch_key, arch_score_col]].drop_duplicates(arch_key)
    scoring_df = scoring_df.merge(arch_merge, left_on='gtdb_species_clade_id',
                                  right_on=arch_key, how='left', suffixes=('', '_arch'))
    scoring_df.rename(columns={arch_score_col: 'core_fraction'}, inplace=True)
    print(f'Core fraction: mean={scoring_df["core_fraction"].mean():.3f}')
else:
    # Derive from marker matrix if available: is_core columns
    print('Will derive core fraction from species_markers if available.')
    scoring_df['core_fraction'] = np.nan

In [ ]:
# ---------------------------------------------------------------
# 2f. Co-occurrence / complementarity score from NB06
# ---------------------------------------------------------------
# Average complementarity score for each species when paired with
# known PGPB partners.

print('complementarity_network columns:', list(comp_network.columns))

# Identify species/genus and score columns
score_cols = [c for c in comp_network.columns
              if 'score' in c.lower() or 'weight' in c.lower()
              or 'complement' in c.lower() or 'jaccard' in c.lower()]
print(f'Score candidate columns: {score_cols}')

# Identify node columns (source/target or species_1/species_2)
node_col_candidates = [('source', 'target'), ('species_1', 'species_2'),
                        ('genus_1', 'genus_2'), ('node_1', 'node_2')]
node_cols = None
for a, b in node_col_candidates:
    if a in comp_network.columns and b in comp_network.columns:
        node_cols = (a, b)
        break
print(f'Network node columns: {node_cols}')

if score_cols and node_cols:
    net_score_col = score_cols[0]
    src, tgt = node_cols
    # Average complementarity per species across all partners
    avg_comp_src = comp_network.groupby(src)[net_score_col].mean().reset_index()
    avg_comp_src.columns = ['species_key', 'avg_complementarity']
    avg_comp_tgt = comp_network.groupby(tgt)[net_score_col].mean().reset_index()
    avg_comp_tgt.columns = ['species_key', 'avg_complementarity']
    avg_comp = pd.concat([avg_comp_src, avg_comp_tgt]).groupby('species_key')['avg_complementarity'].mean().reset_index()

    # Try to join on species id or genus
    scoring_df = scoring_df.merge(avg_comp, left_on='gtdb_species_clade_id',
                                  right_on='species_key', how='left')
    scoring_df.drop(columns=['species_key'], errors='ignore', inplace=True)
    n_matched = scoring_df['avg_complementarity'].notna().sum()
    if n_matched == 0:
        # Try join on genus instead
        scoring_df.drop(columns=['avg_complementarity'], inplace=True)
        scoring_df = scoring_df.merge(avg_comp, left_on='genus',
                                      right_on='species_key', how='left')
        scoring_df.drop(columns=['species_key'], errors='ignore', inplace=True)
        n_matched = scoring_df['avg_complementarity'].notna().sum()
    print(f'Complementarity scores matched: {n_matched:,} species')
else:
    scoring_df['avg_complementarity'] = np.nan
    print('No complementarity scores available; setting to NaN.')

In [ ]:
# ---------------------------------------------------------------
# 2g. Metabolic score integration
# ---------------------------------------------------------------
# Merge metabolic breadth from genus_profiles

genus_key = None
for candidate in ['genus', 'gtdb_species_clade_id', 'species']:
    if candidate in genus_profiles.columns:
        genus_key = candidate
        break
print(f'genus_profiles join key: {genus_key}')

if metabolic_col and genus_key:
    met_merge = genus_profiles[[genus_key, metabolic_col]].drop_duplicates(genus_key)
    scoring_df = scoring_df.merge(met_merge, left_on='genus',
                                  right_on=genus_key, how='left',
                                  suffixes=('', '_gp'))
    scoring_df.rename(columns={metabolic_col: 'metabolic_breadth'}, inplace=True)
    print(f'Metabolic breadth matched: {scoring_df["metabolic_breadth"].notna().sum():,}')
else:
    # Derive metabolic breadth from total marker count
    scoring_df['metabolic_breadth'] = scoring_df['n_pgp_functions'] + scoring_df['n_pathogen_functions']
    print('Metabolic breadth derived from marker function count.')

print(f'\nScoring dataframe shape: {scoring_df.shape}')
print(scoring_df[['pgp_score_raw', 'pathogen_score_raw', 'core_fraction',
                   'avg_complementarity', 'metabolic_breadth']].describe().round(3))

In [ ]:
# ---------------------------------------------------------------
# 2h. Normalize each sub-score to [0, 1] and compute composite
# ---------------------------------------------------------------

def normalize_01(series):
    """Min-max normalize to [0, 1], handling NaN and constant columns."""
    s = series.copy()
    lo, hi = s.min(), s.max()
    if hi == lo:
        return s.fillna(0) * 0  # constant -> 0
    return ((s - lo) / (hi - lo)).fillna(0)

scoring_df['pgp_norm'] = normalize_01(scoring_df['pgp_score_raw'])
scoring_df['pathogen_norm'] = normalize_01(scoring_df['pathogen_score_raw'])
scoring_df['core_frac_norm'] = normalize_01(scoring_df['core_fraction'])
scoring_df['complementarity_norm'] = normalize_01(scoring_df['avg_complementarity'])
scoring_df['metabolic_norm'] = normalize_01(scoring_df['metabolic_breadth'])

# Composite PGP score: PGP markers + core stability + complementarity + metabolic
W_PGP_MARKER = 0.40
W_CORE_FRAC  = 0.20
W_COMPLEMENT = 0.15
W_METABOLIC  = 0.10
W_PATH_PEN   = 0.15  # pathogen penalty (subtracted)

scoring_df['composite_pgp'] = (
    W_PGP_MARKER  * scoring_df['pgp_norm']
    + W_CORE_FRAC * scoring_df['core_frac_norm']
    + W_COMPLEMENT * scoring_df['complementarity_norm']
    + W_METABOLIC * scoring_df['metabolic_norm']
)

# Composite Pathogen score: pathogen markers + inverse core stability
W_PATH_MARKER = 0.50
W_ACCESSORY   = 0.20  # 1 - core fraction (mobile/accessory = more pathogenic)
W_PGP_OFFSET  = 0.15  # PGP offsets pathogenicity
W_MET_PATH    = 0.15

scoring_df['composite_pathogen'] = (
    W_PATH_MARKER * scoring_df['pathogen_norm']
    + W_ACCESSORY * (1 - scoring_df['core_frac_norm'])
    + W_MET_PATH  * scoring_df['metabolic_norm']
)

print('Composite score distributions:')
print(scoring_df[['composite_pgp', 'composite_pathogen']].describe().round(3))

## 3. Cohort assignment

Four cohorts based on composite scores:
- **beneficial**: composite PGP exceeds pathogen by threshold
- **pathogenic**: composite pathogen exceeds PGP by threshold
- **dual_nature**: both PGP and pathogen scores above their respective thresholds
- **neutral**: neither score above threshold

In [ ]:
# Thresholds: species must exceed these to be classified as non-neutral
# Use median of non-zero scores as adaptive threshold
pgp_nonzero = scoring_df.loc[scoring_df['pgp_score_raw'] > 0, 'composite_pgp']
path_nonzero = scoring_df.loc[scoring_df['pathogen_score_raw'] > 0, 'composite_pathogen']

PGP_THRESHOLD = pgp_nonzero.median() if len(pgp_nonzero) > 0 else 0.1
PATH_THRESHOLD = path_nonzero.median() if len(path_nonzero) > 0 else 0.1

print(f'Adaptive thresholds:')
print(f'  PGP threshold:      {PGP_THRESHOLD:.3f} (median of non-zero composite PGP)')
print(f'  Pathogen threshold: {PATH_THRESHOLD:.3f} (median of non-zero composite pathogen)')


def assign_cohort(row):
    """Assign cohort based on composite PGP and pathogen scores."""
    above_pgp = row['composite_pgp'] >= PGP_THRESHOLD
    above_path = row['composite_pathogen'] >= PATH_THRESHOLD

    if above_pgp and above_path:
        return 'dual_nature'
    elif above_pgp:
        return 'beneficial'
    elif above_path:
        return 'pathogenic'
    else:
        return 'neutral'


scoring_df['cohort'] = scoring_df.apply(assign_cohort, axis=1)

print(f'\nCohort distribution:')
cohort_counts = scoring_df['cohort'].value_counts()
for cohort, n in cohort_counts.items():
    pct = 100 * n / len(scoring_df)
    print(f'  {cohort:15s}: {n:6,} species ({pct:.1f}%)')

In [ ]:
# Quick sanity check: composite score scatter
fig, ax = plt.subplots(figsize=(8, 7))

cohort_palette = {
    'beneficial': '#4CAF50',
    'pathogenic': '#F44336',
    'dual_nature': '#FF9800',
    'neutral': '#BDBDBD',
}

for cohort, color in cohort_palette.items():
    sub = scoring_df[scoring_df['cohort'] == cohort]
    ax.scatter(sub['composite_pgp'], sub['composite_pathogen'],
              c=color, label=f'{cohort} (n={len(sub):,})',
              alpha=0.4, s=10, edgecolors='none')

ax.axhline(PATH_THRESHOLD, color='#F44336', linestyle='--', alpha=0.5, linewidth=0.8)
ax.axvline(PGP_THRESHOLD, color='#4CAF50', linestyle='--', alpha=0.5, linewidth=0.8)
ax.set_xlabel('Composite PGP score')
ax.set_ylabel('Composite pathogen score')
ax.set_title('Species cohort classification')
ax.legend(fontsize=9, markerscale=3)
plt.tight_layout()
plt.show()

## 4. Validation against BacDive phenotypes and literature

In [ ]:
# ---------------------------------------------------------------
# 4a. Validate known PGPB genera
# ---------------------------------------------------------------
KNOWN_PGPB = [
    'Rhizobium', 'Bradyrhizobium', 'Mesorhizobium', 'Sinorhizobium',
    'Azospirillum', 'Azotobacter', 'Herbaspirillum',
    'Bacillus',   # B. subtilis, B. amyloliquefaciens
    'Pseudomonas', # P. fluorescens, P. putida
    'Burkholderia',
    'Paenibacillus',
    'Gluconacetobacter',
]

KNOWN_PATHOGENS = [
    'Ralstonia',       # R. solanacearum
    'Xanthomonas',     # X. campestris, X. oryzae
    'Pectobacterium',  # soft rot
    'Dickeya',         # soft rot
    'Erwinia',         # fire blight
    'Agrobacterium',   # crown gall
    'Pseudomonas',     # P. syringae
    'Clavibacter',     # ring rot
]

print('=== Validation: Known PGPB Genera ===')
print(f'{"Genus":25s} {"N_spp":>6s} {"beneficial":>10s} {"dual":>6s} {"pathogenic":>10s} {"neutral":>8s} {"Agree%":>8s}')
print('-' * 80)

pgpb_hits = 0
pgpb_total = 0
for genus in KNOWN_PGPB:
    sub = scoring_df[scoring_df['genus'] == genus]
    if len(sub) == 0:
        # Try partial match
        sub = scoring_df[scoring_df['genus'].str.contains(genus, na=False)]
    if len(sub) == 0:
        continue
    counts = sub['cohort'].value_counts()
    n_ben = counts.get('beneficial', 0) + counts.get('dual_nature', 0)
    agree_pct = 100 * n_ben / len(sub)
    pgpb_hits += n_ben
    pgpb_total += len(sub)
    print(f'{genus:25s} {len(sub):6d} {counts.get("beneficial", 0):10d} '
          f'{counts.get("dual_nature", 0):6d} {counts.get("pathogenic", 0):10d} '
          f'{counts.get("neutral", 0):8d} {agree_pct:7.1f}%')

if pgpb_total > 0:
    print(f'\nOverall PGPB agreement: {pgpb_hits}/{pgpb_total} = {100*pgpb_hits/pgpb_total:.1f}%')

In [ ]:
# ---------------------------------------------------------------
# 4b. Validate known plant pathogens
# ---------------------------------------------------------------

print('=== Validation: Known Plant Pathogen Genera ===')
print(f'{"Genus":25s} {"N_spp":>6s} {"pathogenic":>10s} {"dual":>6s} {"beneficial":>10s} {"neutral":>8s} {"Agree%":>8s}')
print('-' * 80)

path_hits = 0
path_total = 0
for genus in KNOWN_PATHOGENS:
    sub = scoring_df[scoring_df['genus'] == genus]
    if len(sub) == 0:
        sub = scoring_df[scoring_df['genus'].str.contains(genus, na=False)]
    if len(sub) == 0:
        continue
    counts = sub['cohort'].value_counts()
    n_path = counts.get('pathogenic', 0) + counts.get('dual_nature', 0)
    agree_pct = 100 * n_path / len(sub)
    path_hits += n_path
    path_total += len(sub)
    print(f'{genus:25s} {len(sub):6d} {counts.get("pathogenic", 0):10d} '
          f'{counts.get("dual_nature", 0):6d} {counts.get("beneficial", 0):10d} '
          f'{counts.get("neutral", 0):8d} {agree_pct:7.1f}%')

if path_total > 0:
    print(f'\nOverall pathogen agreement: {path_hits}/{path_total} = {100*path_hits/path_total:.1f}%')

# Combined validation rate
total_validated = pgpb_hits + path_hits
total_tested = pgpb_total + path_total
if total_tested > 0:
    print(f'\n=== Combined validation rate: {total_validated}/{total_tested} = '
          f'{100*total_validated/total_tested:.1f}% ===')

## 5. Deep-dive dossiers for top 20–30 genera

In [ ]:
# ---------------------------------------------------------------
# Build genus-level dossiers with comprehensive profiles
# ---------------------------------------------------------------

# Genus-level aggregation
genus_agg = scoring_df.groupby('genus').agg(
    n_species=('gtdb_species_clade_id', 'nunique'),
    mean_pgp_score=('pgp_score_raw', 'mean'),
    mean_pathogen_score=('pathogen_score_raw', 'mean'),
    mean_composite_pgp=('composite_pgp', 'mean'),
    mean_composite_pathogen=('composite_pathogen', 'mean'),
    mean_core_fraction=('core_fraction', 'mean'),
    mean_complementarity=('avg_complementarity', 'mean'),
    n_pgp_functions_mean=('n_pgp_functions', 'mean'),
    n_pathogen_functions_mean=('n_pathogen_functions', 'mean'),
).reset_index()

# Dominant cohort per genus (majority vote)
genus_cohort = scoring_df.groupby(['genus', 'cohort']).size().reset_index(name='count')
idx = genus_cohort.groupby('genus')['count'].idxmax()
genus_dom_cohort = genus_cohort.loc[idx][['genus', 'cohort']].rename(
    columns={'cohort': 'dominant_cohort'}
)
genus_agg = genus_agg.merge(genus_dom_cohort, on='genus', how='left')

# Dominant compartment per genus
genus_compartment = scoring_df.groupby(['genus', 'dominant_compartment']).size().reset_index(name='count')
idx2 = genus_compartment.groupby('genus')['count'].idxmax()
genus_dom_comp = genus_compartment.loc[idx2][['genus', 'dominant_compartment']].rename(
    columns={'dominant_compartment': 'primary_compartment'}
)
genus_agg = genus_agg.merge(genus_dom_comp, on='genus', how='left')

# Determine which PGP/pathogen markers each genus has
# Re-derive from species_markers
marker_cols_pgp = [c for c in species_markers.columns if c in {
    'nitrogen_fixation', 'siderophore', 'acc_deaminase',
    'phosphate_solubilization', 'iaa_biosynthesis', 'hydrogen_cyanide',
    'acetoin_butanediol', 'dapg_biocontrol', 'phenazine', 'biofilm',
}]
marker_cols_path = [c for c in species_markers.columns if c in {
    't3ss', 't4ss', 't6ss', 'effector', 'cwde_pectinase', 'cwde_cellulase',
    'coronatine_toxin', 'phytotoxin', 't2ss',
    't3ss_needle', 't3ss_prgH', 't3ss_secretin', 't3ss_product',
    't6ss_hcp', 't6ss_vgrg', 't6ss_product',
    'virb8_t4ss', 'pectate_lyase', 'pectate_lyase_3', 'cellulase',
}]

# Merge genus from cohort_markers into species_markers
sm_with_genus = species_markers.merge(
    cohort_markers[['gtdb_species_clade_id', 'genus']].drop_duplicates('gtdb_species_clade_id'),
    on='gtdb_species_clade_id', how='left'
)

# For each genus, list which PGP markers are present in > 50% of species
def summarize_markers(group, marker_cols):
    """Return comma-separated list of markers present in >50% of group species."""
    if len(group) == 0:
        return ''
    present = []
    for col in marker_cols:
        if col in group.columns:
            frac = (group[col] > 0).mean()
            if frac > 0.5:
                present.append(col)
    return ', '.join(present) if present else 'none'

pgp_summary = sm_with_genus.groupby('genus').apply(
    lambda g: summarize_markers(g, marker_cols_pgp)
).reset_index(name='pgp_markers')

path_summary = sm_with_genus.groupby('genus').apply(
    lambda g: summarize_markers(g, marker_cols_path)
).reset_index(name='pathogen_markers')

genus_agg = genus_agg.merge(pgp_summary, on='genus', how='left')
genus_agg = genus_agg.merge(path_summary, on='genus', how='left')

print(f'Genus-level dossiers: {len(genus_agg):,} genera')
print(f'Cohort distribution:')
print(genus_agg['dominant_cohort'].value_counts().to_string())

In [ ]:
# ---------------------------------------------------------------
# Select top 20-30 genera for deep-dive dossiers
# ---------------------------------------------------------------
# Strategy: top genera by species count per cohort, ensuring representation

dossier_genera = []
for cohort_name in ['beneficial', 'pathogenic', 'dual_nature', 'neutral']:
    sub = genus_agg[genus_agg['dominant_cohort'] == cohort_name]
    n_pick = {'beneficial': 10, 'pathogenic': 8, 'dual_nature': 7, 'neutral': 5}[cohort_name]
    top = sub.nlargest(n_pick, 'n_species')
    dossier_genera.append(top)

dossier_df = pd.concat(dossier_genera).drop_duplicates('genus')
print(f'Selected {len(dossier_df)} genera for deep-dive dossiers')
print(f'  by cohort: {dossier_df["dominant_cohort"].value_counts().to_dict()}')

# Display top genera
display_cols = ['genus', 'dominant_cohort', 'primary_compartment', 'n_species',
                'mean_pgp_score', 'mean_pathogen_score', 'mean_core_fraction',
                'pgp_markers', 'pathogen_markers']
display_cols = [c for c in display_cols if c in dossier_df.columns]
print('\n' + dossier_df[display_cols].to_string(index=False))

In [ ]:
# ---------------------------------------------------------------
# Dual-nature assessment for each dossier genus
# ---------------------------------------------------------------
# Classify the nature of dual-nature genera: which capabilities co-occur?

def dual_nature_assessment(row):
    """Assess whether a genus is truly dual-nature and characterize the duality."""
    pgp_m = str(row.get('pgp_markers', ''))
    path_m = str(row.get('pathogen_markers', ''))
    if pgp_m not in ('', 'none', 'nan') and path_m not in ('', 'none', 'nan'):
        return f'DUAL: PGP({pgp_m}) + PATH({path_m})'
    elif pgp_m not in ('', 'none', 'nan'):
        return 'PGP-dominant'
    elif path_m not in ('', 'none', 'nan'):
        return 'PATH-dominant'
    else:
        return 'no strong markers'

dossier_df = dossier_df.copy()
dossier_df['dual_assessment'] = dossier_df.apply(dual_nature_assessment, axis=1)

# Print dual-nature genera specifically
dual_genera = dossier_df[dossier_df['dominant_cohort'] == 'dual_nature']
if len(dual_genera) > 0:
    print('=== Dual-Nature Genera Deep Dive ===')
    for _, row in dual_genera.iterrows():
        print(f'\n{row["genus"]} ({row["n_species"]} species, compartment: {row.get("primary_compartment", "?")})')
        print(f'  PGP score:      {row["mean_pgp_score"]:.1f}')
        print(f'  Pathogen score:  {row["mean_pathogen_score"]:.1f}')
        print(f'  Core fraction:   {row["mean_core_fraction"]:.3f}' if pd.notna(row['mean_core_fraction']) else '  Core fraction:   N/A')
        print(f'  Assessment:      {row["dual_assessment"]}')
else:
    print('No dual-nature genera in the dossier selection.')

## 6. Mechanism hypothesis table

In [ ]:
# ---------------------------------------------------------------
# Generate mechanism hypothesis table by cohort
# ---------------------------------------------------------------

mechanism_rows = []

# --- Beneficial cohort ---
ben_df = scoring_df[scoring_df['cohort'] == 'beneficial']
if len(ben_df) > 0:
    ben_pgp_mean = ben_df['pgp_score_raw'].mean()
    ben_core_mean = ben_df['core_fraction'].mean()
    mechanism_rows.extend([
        {'cohort': 'beneficial', 'mechanism': 'Nitrogen fixation',
         'evidence': f'nifHDK present in PGP-scoring species',
         'confidence': 'high'},
        {'cohort': 'beneficial', 'mechanism': 'Siderophore-mediated iron acquisition',
         'evidence': f'ent/pvd gene clusters; mean PGP score = {ben_pgp_mean:.1f}',
         'confidence': 'high'},
        {'cohort': 'beneficial', 'mechanism': 'ACC deaminase stress relief',
         'evidence': f'acdS prevalence in beneficial cohort',
         'confidence': 'medium'},
        {'cohort': 'beneficial', 'mechanism': 'Biocontrol via antimicrobials',
         'evidence': 'DAPG/phenazine gene clusters in Pseudomonas/Bacillus',
         'confidence': 'medium'},
    ])

# --- Pathogenic cohort ---
path_df = scoring_df[scoring_df['cohort'] == 'pathogenic']
if len(path_df) > 0:
    mechanism_rows.extend([
        {'cohort': 'pathogenic', 'mechanism': 'T3SS effector injection',
         'evidence': 'hrc/hrp + effector genes; hallmark of Pseudomonas syringae, Xanthomonas',
         'confidence': 'high'},
        {'cohort': 'pathogenic', 'mechanism': 'Cell wall degradation (CWDE)',
         'evidence': 'Pectate lyases + cellulases in Pectobacterium/Dickeya',
         'confidence': 'high'},
        {'cohort': 'pathogenic', 'mechanism': 'Phytotoxin production',
         'evidence': 'Coronatine biosynthesis genes',
         'confidence': 'medium'},
    ])

# --- Dual-nature cohort ---
dual_df = scoring_df[scoring_df['cohort'] == 'dual_nature']
if len(dual_df) > 0:
    mechanism_rows.extend([
        {'cohort': 'dual_nature', 'mechanism': 'Context-dependent T6SS',
         'evidence': 'T6SS used for both inter-bacterial competition (biocontrol) and pathogenicity',
         'confidence': 'medium'},
        {'cohort': 'dual_nature', 'mechanism': 'PGP + secretion system co-occurrence',
         'evidence': 'Siderophore + T3SS/T4SS in same genomes (e.g., Burkholderia)',
         'confidence': 'medium'},
        {'cohort': 'dual_nature', 'mechanism': 'Host-range dependent lifestyle switching',
         'evidence': 'Same genus with beneficial and pathogenic species on different hosts',
         'confidence': 'low'},
    ])

# --- Interaction hypotheses ---
mechanism_rows.extend([
    {'cohort': 'interaction', 'mechanism': 'PGPB-pathogen niche exclusion',
     'evidence': 'Negative co-occurrence between beneficial and pathogenic genera in same compartment',
     'confidence': 'medium'},
    {'cohort': 'interaction', 'mechanism': 'Metabolic complementarity facilitation',
     'evidence': 'High complementarity scores between PGPB genera (NB06)',
     'confidence': 'medium'},
    {'cohort': 'interaction', 'mechanism': 'Siderophore-mediated competition',
     'evidence': 'Iron-scavenging beneficial genera may suppress pathogens',
     'confidence': 'low'},
])

mechanism_table = pd.DataFrame(mechanism_rows)

print('=== Mechanism Hypothesis Table ===')
print(mechanism_table.to_string(index=False))

## 7. Hypothesis summary table (H1–H5)

In [ ]:
# ---------------------------------------------------------------
# Compile hypothesis-level evidence summary
# ---------------------------------------------------------------

PLANT_COMPARTMENTS = {'rhizosphere', 'root', 'phyllosphere', 'endophyte', 'plant_other'}

# --- H1: Compartment specificity ---
# Evidence: do cohort proportions differ across plant compartments?
plant_species = scoring_df[scoring_df['dominant_compartment'].isin(PLANT_COMPARTMENTS)].copy()
if len(plant_species) > 0:
    h1_ct = pd.crosstab(plant_species['dominant_compartment'], plant_species['cohort'])
    if h1_ct.shape[0] >= 2 and h1_ct.shape[1] >= 2:
        h1_chi2, h1_pval, h1_dof, _ = stats.chi2_contingency(h1_ct)
        h1_supported = 'Supported' if h1_pval < 0.05 else 'Not supported'
        h1_evidence = f'chi2={h1_chi2:.1f}, p={h1_pval:.2e}, {h1_ct.shape[0]} compartments x {h1_ct.shape[1]} cohorts'
    else:
        h1_supported = 'Insufficient data'
        h1_evidence = f'Only {h1_ct.shape[0]} compartments or {h1_ct.shape[1]} cohorts'
        h1_pval = np.nan
else:
    h1_supported = 'No data'
    h1_evidence = 'No plant-associated species with cohort labels'
    h1_pval = np.nan

# --- H2: Core vs accessory architecture ---
# Evidence: do beneficial species have higher core fraction for PGP genes?
ben_core = scoring_df.loc[scoring_df['cohort'] == 'beneficial', 'core_fraction'].dropna()
path_core = scoring_df.loc[scoring_df['cohort'] == 'pathogenic', 'core_fraction'].dropna()
if len(ben_core) >= 5 and len(path_core) >= 5:
    h2_stat, h2_pval = stats.mannwhitneyu(ben_core, path_core, alternative='two-sided')
    h2_supported = 'Supported' if h2_pval < 0.05 else 'Not supported'
    h2_evidence = (f'Beneficial core frac={ben_core.mean():.3f} vs '
                   f'Pathogenic={path_core.mean():.3f}, U={h2_stat:.0f}, p={h2_pval:.2e}')
else:
    h2_supported = 'Insufficient data'
    h2_evidence = f'Beneficial n={len(ben_core)}, Pathogenic n={len(path_core)}'
    h2_pval = np.nan

# --- H3: Metabolic complementarity ---
# Evidence: do beneficial species have higher complementarity scores?
ben_comp = scoring_df.loc[scoring_df['cohort'] == 'beneficial', 'avg_complementarity'].dropna()
path_comp = scoring_df.loc[scoring_df['cohort'] == 'pathogenic', 'avg_complementarity'].dropna()
if len(ben_comp) >= 5 and len(path_comp) >= 5:
    h3_stat, h3_pval = stats.mannwhitneyu(ben_comp, path_comp, alternative='greater')
    h3_supported = 'Supported' if h3_pval < 0.05 else 'Not supported'
    h3_evidence = (f'Beneficial complementarity={ben_comp.mean():.3f} vs '
                   f'Pathogenic={path_comp.mean():.3f}, U={h3_stat:.0f}, p={h3_pval:.2e}')
else:
    h3_supported = 'Insufficient data'
    h3_evidence = f'Beneficial n={len(ben_comp)}, Pathogenic n={len(path_comp)}'
    h3_pval = np.nan

# --- H4: Mobility proxies ---
# Evidence: do pathogenic genes reside more on accessory/singleton clusters?
# Proxy: inverse of core_fraction for pathogen-scored species
if len(path_core) >= 5:
    neutral_core = scoring_df.loc[scoring_df['cohort'] == 'neutral', 'core_fraction'].dropna()
    if len(neutral_core) >= 5:
        h4_stat, h4_pval = stats.mannwhitneyu(path_core, neutral_core, alternative='less')
        h4_supported = 'Supported' if h4_pval < 0.05 else 'Not supported'
        h4_evidence = (f'Pathogenic core frac={path_core.mean():.3f} vs '
                       f'Neutral={neutral_core.mean():.3f}, U={h4_stat:.0f}, p={h4_pval:.2e}')
    else:
        h4_supported = 'Insufficient data'
        h4_evidence = f'Neutral n={len(neutral_core)}'
        h4_pval = np.nan
else:
    h4_supported = 'Insufficient data'
    h4_evidence = f'Pathogenic core fraction n={len(path_core)}'
    h4_pval = np.nan

# --- H5: Novel markers ---
h5_n_novel = len(novel_markers)
h5_supported = 'Supported' if h5_n_novel > 0 else 'Not supported'
h5_evidence = f'{h5_n_novel} novel plant-enriched OGs identified (NB03)'

# Compile
hypotheses = pd.DataFrame([
    {'hypothesis': 'H1: Compartment specificity',
     'status': h1_supported, 'key_evidence': h1_evidence},
    {'hypothesis': 'H2: Core vs accessory architecture',
     'status': h2_supported, 'key_evidence': h2_evidence},
    {'hypothesis': 'H3: Metabolic complementarity',
     'status': h3_supported, 'key_evidence': h3_evidence},
    {'hypothesis': 'H4: Mobility proxies',
     'status': h4_supported, 'key_evidence': h4_evidence},
    {'hypothesis': 'H5: Novel markers',
     'status': h5_supported, 'key_evidence': h5_evidence},
])

print('=' * 100)
print('HYPOTHESIS SUMMARY TABLE')
print('=' * 100)
for _, row in hypotheses.iterrows():
    print(f'\n{row["hypothesis"]}')
    print(f'  Status:   {row["status"]}')
    print(f'  Evidence: {row["key_evidence"]}')
print('\n' + '=' * 100)

## 8. Summary figures (3 panels)

- **Panel A**: Genus heatmap (top 30 genera x key markers), colored by cohort
- **Panel B**: Compartment x cohort distribution (stacked bar)
- **Panel C**: Architecture comparison (core fraction: beneficial vs pathogenic vs dual)

In [ ]:
# ---------------------------------------------------------------
# Panel A: Genus heatmap of key markers
# ---------------------------------------------------------------

# Select top 30 genera by species count from dossier_df
top_genera = dossier_df.nlargest(30, 'n_species')['genus'].tolist()

# Build heatmap matrix: genus x marker category
# Use species_markers merged with genus
all_marker_cols = marker_cols_pgp + marker_cols_path
all_marker_cols = [c for c in all_marker_cols if c in sm_with_genus.columns]

# If no specific marker columns found, use available functional categories
if len(all_marker_cols) == 0:
    # Fall back to presence columns
    all_marker_cols = [c for c in sm_with_genus.columns
                       if c.endswith('_present') and c != 'is_plant_associated']
    print(f'Using _present columns: {len(all_marker_cols)} markers')
else:
    print(f'Using raw marker columns: {len(all_marker_cols)} markers')

heatmap_data = sm_with_genus[sm_with_genus['genus'].isin(top_genera)].groupby('genus')[
    all_marker_cols
].apply(lambda g: (g > 0).mean())  # fraction of species with each marker

# Reindex to keep order
heatmap_data = heatmap_data.reindex(top_genera)

# Color map for cohort row annotation
genus_to_cohort = dict(zip(dossier_df['genus'], dossier_df['dominant_cohort']))

print(f'Heatmap: {heatmap_data.shape[0]} genera x {heatmap_data.shape[1]} markers')

In [ ]:
# ---------------------------------------------------------------
# Panels B & C preparation
# ---------------------------------------------------------------

# Panel B: compartment x cohort crosstab
comp_cohort_ct = pd.crosstab(
    scoring_df['dominant_compartment'],
    scoring_df['cohort']
)
# Focus on plant compartments + soil for contrast
comp_order = ['rhizosphere', 'root', 'phyllosphere', 'endophyte', 'plant_other', 'soil']
comp_cohort_ct = comp_cohort_ct.reindex([c for c in comp_order if c in comp_cohort_ct.index])

# Panel C: core fraction by cohort
arch_data = scoring_df[scoring_df['cohort'].isin(['beneficial', 'pathogenic', 'dual_nature'])].copy()
arch_data = arch_data[arch_data['core_fraction'].notna()]

print(f'Panel B: {comp_cohort_ct.shape[0]} compartments x {comp_cohort_ct.shape[1]} cohorts')
print(f'Panel C: {len(arch_data):,} species with core_fraction data')

In [ ]:
# ---------------------------------------------------------------
# Generate synthesis_overview.png (3-panel figure)
# ---------------------------------------------------------------

fig = plt.figure(figsize=(20, 7))
gs = fig.add_gridspec(1, 3, width_ratios=[1.5, 1, 1], wspace=0.35)

cohort_colors = {
    'beneficial': '#4CAF50',
    'pathogenic': '#F44336',
    'dual_nature': '#FF9800',
    'neutral': '#BDBDBD',
}

# --- Panel A: Genus heatmap ---
ax_a = fig.add_subplot(gs[0])

if heatmap_data.shape[0] > 0 and heatmap_data.shape[1] > 0:
    # Truncate column names for display
    col_labels = [c.replace('_present', '').replace('_', ' ')[:15] for c in heatmap_data.columns]

    im = ax_a.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd',
                     vmin=0, vmax=1, interpolation='nearest')
    ax_a.set_yticks(range(len(heatmap_data)))
    ax_a.set_yticklabels(heatmap_data.index, fontsize=7)
    ax_a.set_xticks(range(len(col_labels)))
    ax_a.set_xticklabels(col_labels, fontsize=6, rotation=60, ha='right')
    ax_a.set_title('A. Genus marker profile (fraction of species)', fontsize=11)

    # Add cohort color sidebar
    for i, genus in enumerate(heatmap_data.index):
        cohort = genus_to_cohort.get(genus, 'neutral')
        ax_a.add_patch(plt.Rectangle((-1.5, i - 0.5), 0.8, 1,
                       color=cohort_colors.get(cohort, '#BDBDBD'),
                       clip_on=False))

    cb = plt.colorbar(im, ax=ax_a, shrink=0.6, pad=0.02)
    cb.set_label('Fraction of species', fontsize=8)
else:
    ax_a.text(0.5, 0.5, 'Insufficient heatmap data',
             ha='center', va='center', transform=ax_a.transAxes)
    ax_a.set_title('A. Genus marker profile', fontsize=11)

# --- Panel B: Compartment x Cohort stacked bar ---
ax_b = fig.add_subplot(gs[1])

if comp_cohort_ct.shape[0] > 0:
    # Normalize to proportions
    comp_cohort_pct = comp_cohort_ct.div(comp_cohort_ct.sum(axis=1), axis=0) * 100

    cohort_order = ['beneficial', 'dual_nature', 'pathogenic', 'neutral']
    cohort_order = [c for c in cohort_order if c in comp_cohort_pct.columns]

    bottom = np.zeros(len(comp_cohort_pct))
    for cohort in cohort_order:
        if cohort in comp_cohort_pct.columns:
            vals = comp_cohort_pct[cohort].values
            ax_b.barh(range(len(comp_cohort_pct)), vals, left=bottom,
                     color=cohort_colors.get(cohort, '#9E9E9E'),
                     label=cohort, edgecolor='white', linewidth=0.5)
            bottom += vals

    ax_b.set_yticks(range(len(comp_cohort_pct)))
    ax_b.set_yticklabels(comp_cohort_pct.index, fontsize=9)
    ax_b.set_xlabel('Percentage of species', fontsize=9)
    ax_b.legend(fontsize=7, loc='lower right')
else:
    ax_b.text(0.5, 0.5, 'Insufficient compartment data',
             ha='center', va='center', transform=ax_b.transAxes)
ax_b.set_title('B. Cohort distribution by compartment', fontsize=11)

# --- Panel C: Core fraction comparison ---
ax_c = fig.add_subplot(gs[2])

cohort_order_c = ['beneficial', 'pathogenic', 'dual_nature']
box_data = []
box_labels = []
box_colors = []
for cohort in cohort_order_c:
    vals = arch_data.loc[arch_data['cohort'] == cohort, 'core_fraction'].dropna()
    if len(vals) > 0:
        box_data.append(vals.values)
        box_labels.append(f'{cohort}\n(n={len(vals):,})')
        box_colors.append(cohort_colors.get(cohort, '#9E9E9E'))

if box_data:
    bp = ax_c.boxplot(box_data, labels=box_labels, patch_artist=True,
                      widths=0.6, showfliers=False)
    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # Add strip plot
    for i, (data, color) in enumerate(zip(box_data, box_colors), 1):
        jitter = np.random.normal(0, 0.04, size=len(data))
        ax_c.scatter(np.full_like(data, i) + jitter, data,
                    alpha=0.15, s=5, c=color, edgecolors='none')

    ax_c.set_ylabel('Core gene fraction', fontsize=9)

    # Add significance annotation if H2 was tested
    if 'h2_pval' in dir() and not np.isnan(h2_pval):
        sig_label = f'p = {h2_pval:.1e}' if h2_pval < 0.001 else f'p = {h2_pval:.3f}'
        # Draw bracket between beneficial and pathogenic
        y_max = max(max(d) for d in box_data) * 1.05
        ax_c.plot([1, 1, 2, 2], [y_max, y_max * 1.02, y_max * 1.02, y_max],
                 color='black', linewidth=0.8)
        ax_c.text(1.5, y_max * 1.03, sig_label, ha='center', fontsize=8)
else:
    ax_c.text(0.5, 0.5, 'No core fraction data\navailable',
             ha='center', va='center', transform=ax_c.transAxes, fontsize=10)
ax_c.set_title('C. Genomic architecture by cohort', fontsize=11)

plt.savefig(os.path.join(FIGURES, 'synthesis_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved {os.path.join(FIGURES, "synthesis_overview.png")}')

## 9. PGP–Pathogen interaction summary

In [ ]:
# ---------------------------------------------------------------
# 9a. Dual-nature genera profile
# ---------------------------------------------------------------

dual_species = scoring_df[scoring_df['cohort'] == 'dual_nature'].copy()

print('=== Dual-Nature Species Profile ===')
print(f'Total dual-nature species: {len(dual_species):,}')

if len(dual_species) > 0:
    print(f'\nCompartment distribution:')
    print(dual_species['dominant_compartment'].value_counts().to_string())

    print(f'\nTop genera:')
    print(dual_species['genus'].value_counts().head(15).to_string())

    print(f'\nScore statistics:')
    print(f'  PGP score:      mean={dual_species["pgp_score_raw"].mean():.1f}, '
          f'median={dual_species["pgp_score_raw"].median():.1f}')
    print(f'  Pathogen score:  mean={dual_species["pathogen_score_raw"].mean():.1f}, '
          f'median={dual_species["pathogen_score_raw"].median():.1f}')
    print(f'  Core fraction:   mean={dual_species["core_fraction"].mean():.3f}'
          if dual_species['core_fraction'].notna().any()
          else '  Core fraction:   N/A')

In [ ]:
# ---------------------------------------------------------------
# 9b. Co-occurrence vs exclusion patterns
# ---------------------------------------------------------------

# For each compartment, check if beneficial and pathogenic genera co-occur
# or show mutual exclusion

print('=== Co-occurrence vs Exclusion Patterns ===')
print()

for comp in ['rhizosphere', 'root', 'phyllosphere', 'endophyte']:
    comp_sp = scoring_df[scoring_df['dominant_compartment'] == comp]
    if len(comp_sp) == 0:
        continue

    n_ben = (comp_sp['cohort'] == 'beneficial').sum()
    n_path = (comp_sp['cohort'] == 'pathogenic').sum()
    n_dual = (comp_sp['cohort'] == 'dual_nature').sum()
    n_total = len(comp_sp)

    ben_genera = set(comp_sp.loc[comp_sp['cohort'] == 'beneficial', 'genus'].dropna().unique())
    path_genera = set(comp_sp.loc[comp_sp['cohort'] == 'pathogenic', 'genus'].dropna().unique())
    overlap_genera = ben_genera & path_genera

    print(f'--- {comp} ({n_total} species) ---')
    print(f'  Beneficial: {n_ben} species ({100*n_ben/n_total:.1f}%)')
    print(f'  Pathogenic: {n_path} species ({100*n_path/n_total:.1f}%)')
    print(f'  Dual-nature: {n_dual} species ({100*n_dual/n_total:.1f}%)')
    print(f'  Genera with BOTH beneficial and pathogenic species: {len(overlap_genera)}')
    if overlap_genera:
        print(f'    -> {sorted(overlap_genera)[:10]}')
    print()

In [ ]:
# ---------------------------------------------------------------
# 9c. Compartment-specific interaction summary
# ---------------------------------------------------------------

print('=== Compartment-Specific Interaction Summary ===')
print()

interaction_rows = []
for comp in ['rhizosphere', 'root', 'phyllosphere', 'endophyte']:
    comp_sp = scoring_df[scoring_df['dominant_compartment'] == comp]
    if len(comp_sp) < 10:
        continue

    ben_frac = (comp_sp['cohort'] == 'beneficial').mean()
    path_frac = (comp_sp['cohort'] == 'pathogenic').mean()
    dual_frac = (comp_sp['cohort'] == 'dual_nature').mean()

    # Determine dominant interaction pattern
    if ben_frac > 0.3 and path_frac < 0.1:
        pattern = 'PGPB-dominated; low pathogen pressure'
    elif path_frac > 0.3 and ben_frac < 0.1:
        pattern = 'Pathogen-dominated; limited beneficial community'
    elif dual_frac > 0.15:
        pattern = 'High dual-nature prevalence; context-dependent interactions'
    elif ben_frac > 0.1 and path_frac > 0.1:
        pattern = 'Mixed community; potential competition/coexistence'
    else:
        pattern = 'Neutral-dominated; low functional specialization'

    interaction_rows.append({
        'compartment': comp,
        'n_species': len(comp_sp),
        'ben_frac': f'{ben_frac:.1%}',
        'path_frac': f'{path_frac:.1%}',
        'dual_frac': f'{dual_frac:.1%}',
        'interaction_pattern': pattern,
    })
    print(f'{comp:15s}: {pattern}')

if interaction_rows:
    interaction_summary = pd.DataFrame(interaction_rows)
    print(f'\n{interaction_summary.to_string(index=False)}')

## 10. Save all final outputs

In [ ]:
# ---------------------------------------------------------------
# Save cohort assignments
# ---------------------------------------------------------------
cohort_out_cols = [
    'gtdb_species_clade_id', 'genus', 'dominant_compartment',
    'pgp_score_raw', 'pathogen_score_raw',
    'composite_pgp', 'composite_pathogen',
    'core_fraction', 'avg_complementarity',
    'cohort',
    'n_pgp_functions', 'n_pathogen_functions',
]
cohort_out_cols = [c for c in cohort_out_cols if c in scoring_df.columns]
scoring_df[cohort_out_cols].to_csv(
    os.path.join(DATA, 'cohort_assignments.csv'), index=False
)
print(f'Saved data/cohort_assignments.csv ({len(scoring_df):,} species)')

# ---------------------------------------------------------------
# Save genus dossiers
# ---------------------------------------------------------------
dossier_out_cols = [
    'genus', 'dominant_cohort', 'primary_compartment', 'n_species',
    'mean_pgp_score', 'mean_pathogen_score',
    'mean_composite_pgp', 'mean_composite_pathogen',
    'mean_core_fraction', 'mean_complementarity',
    'pgp_markers', 'pathogen_markers',
    'dual_assessment',
]
dossier_out_cols = [c for c in dossier_out_cols if c in dossier_df.columns]
dossier_df[dossier_out_cols].to_csv(
    os.path.join(DATA, 'genus_dossiers.csv'), index=False
)
print(f'Saved data/genus_dossiers.csv ({len(dossier_df):,} genera)')

# ---------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------
print('\n' + '=' * 70)
print('NB07 SYNTHESIS SUMMARY')
print('=' * 70)
print(f'Total species classified: {len(scoring_df):,}')
print(f'Cohort distribution:')
for cohort, n in scoring_df['cohort'].value_counts().items():
    pct = 100 * n / len(scoring_df)
    print(f'  {cohort:15s}: {n:6,} ({pct:.1f}%)')
print(f'\nValidation:')
if pgpb_total > 0:
    print(f'  Known PGPB agreement:     {100*pgpb_hits/pgpb_total:.1f}% ({pgpb_hits}/{pgpb_total})')
if path_total > 0:
    print(f'  Known pathogen agreement: {100*path_hits/path_total:.1f}% ({path_hits}/{path_total})')
print(f'\nHypothesis outcomes:')
for _, row in hypotheses.iterrows():
    print(f'  {row["hypothesis"]}: {row["status"]}')
print(f'\nGenus dossiers: {len(dossier_df)} genera profiled')
print(f'Novel markers (NB03): {h5_n_novel} OGs')
print(f'\nOutputs saved to {DATA}/')
print('  - cohort_assignments.csv')
print('  - genus_dossiers.csv')
print(f'  - {os.path.join(FIGURES, "synthesis_overview.png")}')
print('\n=== Plant Microbiome Ecotypes project complete ===')